# FINISH ONLY – Productive Cascade + Active Learning + SSL Refresh

Dieses Notebook ist **kein neuer Gesamtlauf**.

Es setzt ausschließlich einen bereits abgeschlossenen/abgebrochenen
`phreshphish_PRODUCTIVE_CASCADE_AL_SSL_LOOP` fort und führt nur den Teil aus, der beim
vorherigen Lauf nach rund 19.000 Sekunden noch fehlte:

1. vorhandene Active-Learning-/Kaskaden-Ergebnisse und Selection-History laden,
2. vorhandene 25-%-Loop-Zustände validieren,
3. den korrigierten **TAPT-like MLM/SSL-Refresh** für 10 Seeds ausführen,
4. DAPT-E2E auf denselben bereits aktiv erworbenen 25-%-Labels neu fine-tunen,
5. direkten Detektor und Kaskade gegen `DAPT_STATIC` vergleichen,
6. Statistiken, Thesis-Tabellen, Abbildungen und ein kleines Analyse-ZIP exportieren.

## Harte Sicherheitsregel

Dieses Notebook **rechnet weder den N10-Finallauf noch den Active-Learning-Loop neu**.
Wenn die benötigten alten Artefakte nicht gefunden werden, bricht es **vor jedem Training**
mit einer Fehlermeldung ab.

Active Learning bleibt supervised Label-Akquisition; der zusätzliche Refresh ist
unlabeled-only MLM und wird deshalb separat als `TAPT-like SSL refresh` ausgewiesen.


In [ ]:

# ============================================================
# 00 – Imports und feste Konfiguration
# ============================================================
import os, gc, json, math, pickle, random, shutil, time, warnings, hashlib, zipfile
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
OUT_ROOT = Path("/kaggle/working/phreshphish_PRODUCTIVE_LOOP_FINISH_ONLY")
SSL_ROOT = OUT_ROOT / "ssl_refresh"
SCORE_ROOT = OUT_ROOT / "ssl_refresh_scores"
TABLE_ROOT = OUT_ROOT / "tables"
FIG_ROOT = OUT_ROOT / "figures"
AUDIT_ROOT = OUT_ROOT / "audit"
for p in [OUT_ROOT, SSL_ROOT, SCORE_ROOT, TABLE_ROOT, FIG_ROOT, AUDIT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

ORIGINAL5 = [42, 52, 62, 72, 82]
REPLICATION5 = [242, 252, 262, 272, 282]
SEEDS = ORIGINAL5 + REPLICATION5

TARGET_FPRS = [0.005, 0.010, 0.020]
PRIMARY_FPR = 0.005
REVIEW_FRACTIONS = [0.05, 0.10, 0.20]
PRIMARY_REVIEW = 0.20
PRIMARY_STRATEGY = "UNCERTAINTY_POST_CASCADE"

CALIBRATION_FRACTION = 0.30
CALIBRATION_SPLIT_SEED = 20260808
AP_BALANCE_SEED = 20260809
MAX_LENGTH = 256

E2E_EPOCHS = 5
E2E_LR = 2e-5
E2E_WEIGHT_DECAY = 0.01
E2E_WARMUP_RATIO = 0.10
E2E_BATCH_CANDIDATES = [16, 8, 4]
E2E_SCORE_BATCH = 64

SSL_REFRESH_EPOCHS = 1
SSL_REFRESH_BATCH = 16
SSL_REFRESH_LR = 1e-5
SSL_REFRESH_WEIGHT_DECAY = 0.01
SSL_REFRESH_MLM_PROB = 0.15

EXPECTED_SCENARIO_ROWS = {
    "TEMPORAL": 8000,
    "DOMAIN_OOD": 3858,
    "TEMPLATE_OOD": 6832,
    "DOMAIN_TEMPLATE_OOD": 3708,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False

print({
    "mode": "FINISH_ONLY",
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "output": str(OUT_ROOT),
})


In [ ]:

# ============================================================
# 01 – Alte Outputs und Inputs automatisch finden
#      FAIL FAST: kein Re-Training des alten Loops erlaubt
# ============================================================
def has_model_files(p):
    return (
        p.is_dir()
        and (p / "config.json").exists()
        and (
            (p / "model.safetensors").exists()
            or (p / "pytorch_model.bin").exists()
        )
    )

def looks_like_loop_resume(p):
    return (
        p.is_dir()
        and (p / "LOOP_results_long.csv").exists()
        and (p / "LOOP_summary.csv").exists()
        and (p / "audit" / "LOOP_selection_history.json").exists()
        and (p / "state_scores").exists()
    )

loop_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_loop_resume(p)]
if looks_like_loop_resume(INPUT_ROOT):
    loop_candidates.append(INPUT_ROOT)

if not loop_candidates:
    raise FileNotFoundError(
        "STOP: Alter produktiver Loop-Output wurde nicht gefunden. "
        "Bitte den Output des ~19.000-s-Laufs als Kaggle-Dataset anhängen. "
        "Benötigt: LOOP_results_long.csv, LOOP_summary.csv, "
        "audit/LOOP_selection_history.json und state_scores/. "
        "Dieses FINISH_ONLY-Notebook startet den alten Loop absichtlich NICHT neu."
    )

# Bevorzugt der vollständigste Output (meiste State-Dateien).
LOOP_SOURCE = max(
    set(loop_candidates),
    key=lambda p: len(list((p / "state_scores").glob("*.npz"))),
)

# N10-Output nur für die fünf Replikations-DAPT-Encoder suchen.
def looks_like_n10(p):
    return (
        p.is_dir()
        and (p / "FINAL_N10_COMPLETE.json").exists()
        and (p / "dapt_encoders").exists()
    )

n10_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_n10(p)]
if looks_like_n10(INPUT_ROOT):
    n10_candidates.append(INPUT_ROOT)
if not n10_candidates:
    raise FileNotFoundError(
        "STOP: Fertiger N10-Output mit FINAL_N10_COMPLETE.json und "
        "dapt_encoders/ fehlt."
    )
N10_SOURCE = max(
    set(n10_candidates),
    key=lambda p: sum(
        has_model_files(p / "dapt_encoders" / f"seed_{s}")
        for s in REPLICATION5
    ),
)

# FINAL-FREEZE nur für Token-Caches.
def looks_like_freeze(p):
    return (
        p.is_dir()
        and (p / "tokens" / "train_4k" / "input_ids.npy").exists()
        and (p / "tokens" / "calibration" / "input_ids.npy").exists()
        and (p / "tokens" / "iid_test" / "input_ids.npy").exists()
        and (p / "tokens" / "holdout_8k" / "input_ids.npy").exists()
    )

freeze_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_freeze(p)]
if looks_like_freeze(INPUT_ROOT):
    freeze_candidates.append(INPUT_ROOT)
if not freeze_candidates:
    raise FileNotFoundError(
        "STOP: FINAL-FREEZE-Input mit tokens/train_4k, calibration, "
        "iid_test und holdout_8k fehlt."
    )
FREEZE_ROOT = sorted(set(freeze_candidates), key=lambda p: len(str(p)))[0]

split_paths = list(INPUT_ROOT.rglob("split_roles_and_holdout_cache_v2_ram_safe.pkl"))
dapt_bundle_paths = list(INPUT_ROOT.rglob("dapt40k_bundle.pkl"))
if not split_paths or not dapt_bundle_paths:
    raise FileNotFoundError("STOP: ENDGAME Split-Cache oder dapt40k_bundle.pkl fehlt.")
SPLIT_PATH = split_paths[0]
DAPT_BUNDLE_PATH = dapt_bundle_paths[0]

base_candidates = [
    p for p in INPUT_ROOT.rglob("roberta-base")
    if has_model_files(p)
]
if not base_candidates:
    raise FileNotFoundError("STOP: lokales roberta-base fehlt.")
BASE_MODEL_DIR = sorted(base_candidates, key=lambda p: len(str(p)))[0]

# DAPT-Encoder für alle 10 Seeds.
DAPT_DIRS = {}
for seed in ORIGINAL5:
    candidates = []
    for p in INPUT_ROOT.rglob(f"seed_{seed}"):
        low = str(p).lower().replace("\\", "/")
        if (
            has_model_files(p)
            and "dapt_encoders" in low
            and "40k" in low
        ):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"STOP: Original-DAPT-40k seed {seed} fehlt.")
    DAPT_DIRS[seed] = sorted(candidates, key=lambda p: len(str(p)))[0]

for seed in REPLICATION5:
    p = N10_SOURCE / "dapt_encoders" / f"seed_{seed}"
    if not has_model_files(p):
        raise FileNotFoundError(f"STOP: N10-DAPT-Replikation seed {seed} fehlt: {p}")
    DAPT_DIRS[seed] = p

print(json.dumps({
    "loop_source": str(LOOP_SOURCE),
    "loop_state_files": len(list((LOOP_SOURCE / "state_scores").glob("*.npz"))),
    "n10_source": str(N10_SOURCE),
    "freeze_root": str(FREEZE_ROOT),
    "base_model": str(BASE_MODEL_DIR),
    "dapt_dirs": {str(k): str(v) for k, v in DAPT_DIRS.items()},
}, indent=2))


In [ ]:

# ============================================================
# 02 – HARTE Resume-Validierung VOR JEDEM TRAINING
# ============================================================
SELECTION_PATH = LOOP_SOURCE / "audit" / "LOOP_selection_history.json"
selection_history = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))

required_small_files = [
    "LOOP_results_long.csv",
    "LOOP_summary.csv",
    "LOOP_statistics_n10.csv",
    "LOOP_acquisition_audit.csv",
    "LOOP_annotation_efficiency.csv",
    "LOOP_T0_DAPT_same_AL_labels.csv",
    "LOOP_T0_DAPT_same_AL_labels_statistics.csv",
]
missing_small = [f for f in required_small_files if not (LOOP_SOURCE / f).exists()]

def static_state_path(seed):
    return (
        LOOP_SOURCE
        / "state_scores"
        / f"DAPT_{PRIMARY_STRATEGY}_seed{seed}_budget0p25.npz"
    )

missing_states = [str(static_state_path(s)) for s in SEEDS if not static_state_path(s).exists()]
bad_selections = []
for seed in SEEDS:
    key = f"{PRIMARY_STRATEGY}__seed{seed}"
    if key not in selection_history:
        bad_selections.append(f"{key}: missing")
        continue
    idx = selection_history[key].get("0.25")
    if idx is None or len(idx) != 1000:
        bad_selections.append(f"{key}: expected 1000 labels, got {None if idx is None else len(idx)}")

if missing_small or missing_states or bad_selections:
    raise RuntimeError(
        "STOP VOR TRAINING – der alte 19.000-s-Output ist nicht vollständig genug.\n"
        f"Fehlende kleine Dateien: {missing_small}\n"
        f"Fehlende 25%-States: {missing_states}\n"
        f"Fehlerhafte Selection-History: {bad_selections}\n"
        "Bitte NICHT weiterlaufen lassen; richtigen alten Loop-Output anhängen."
    )

# Hash-Check: Selection-History muss exakt zu den gespeicherten DAPT-States passen.
hash_mismatches = []
for seed in SEEDS:
    key = f"{PRIMARY_STRATEGY}__seed{seed}"
    idx = np.sort(np.asarray(selection_history[key]["0.25"], dtype=np.int32))
    expected_hash = hashlib.sha256(idx.tobytes()).hexdigest()
    d = np.load(static_state_path(seed), allow_pickle=False)
    stored_hash = str(d["labeled_hash"][0]) if "labeled_hash" in d.files else None
    if stored_hash != expected_hash:
        hash_mismatches.append({
            "seed": seed,
            "expected": expected_hash,
            "stored": stored_hash,
        })

if hash_mismatches:
    raise RuntimeError(
        "STOP VOR TRAINING – Selection-History und gespeicherte Loop-States "
        f"passen nicht zusammen: {hash_mismatches}"
    )

print({
    "resume_validation": "PASS",
    "required_small_files": len(required_small_files),
    "validated_static_states": len(SEEDS),
    "validated_25pct_selection_histories": len(SEEDS),
    "old_loop_retraining": False,
    "n10_retraining": False,
})


In [ ]:

# ============================================================
# 03 – Datenrollen, Szenarien und Token-Caches laden
# ============================================================
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def first_existing(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        if k in obj:
            return obj[k]
    return None

split_payload = load_pickle(SPLIT_PATH)
dapt_payload = load_pickle(DAPT_BUNDLE_PATH)

train_df = first_existing(
    split_payload, ["train_df", "train", "supervised_train", "downstream_train"]
)
val_df = first_existing(
    split_payload, ["val_df", "validation_df", "validation", "val"]
)
holdout_df = first_existing(
    split_payload, ["final_holdout_clean", "final_holdout", "holdout_df", "holdout"]
)
pretrain_df = first_existing(
    dapt_payload, ["pretrain_large_df", "frame", "pretrain_df", "pretrain"]
)
dapt_holdout = first_existing(dapt_payload, ["final_holdout_clean"])

for name, obj in [
    ("train", train_df),
    ("validation", val_df),
    ("holdout", holdout_df),
    ("pretrain40k", pretrain_df),
]:
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"{name}: kein DataFrame gefunden.")

train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
holdout_df = holdout_df.reset_index(drop=True).copy()
pretrain_df = pretrain_df.reset_index(drop=True).copy()

if isinstance(dapt_holdout, pd.DataFrame):
    dapt_holdout = dapt_holdout.reset_index(drop=True)
    lookup = dapt_holdout.copy()
    lookup.index = lookup["sha256"].astype(str)
    hs = holdout_df["sha256"].astype(str)
    for c in [
        "near_duplicate_to_development",
        "min_simhash_distance_to_development",
        "template_seen_in_development",
    ]:
        if c in lookup.columns:
            mapped = hs.map(lookup[c])
            if mapped.isna().any():
                raise RuntimeError(f"Holdout-Flag {c} konnte nicht vollständig gemappt werden.")
            holdout_df[c] = mapped.to_numpy()

cal_idx, iid_idx = train_test_split(
    np.arange(len(val_df)),
    test_size=1.0 - CALIBRATION_FRACTION,
    random_state=CALIBRATION_SPLIT_SEED,
    stratify=val_df["label"].to_numpy(),
)
calibration_df = val_df.iloc[np.sort(cal_idx)].reset_index(drop=True)
iid_df = val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)

development_domains = set()
development_templates = set()
for frame in [pretrain_df, train_df, val_df]:
    development_domains.update(frame["domain"].fillna("").astype(str).tolist())
    development_templates.update(frame["template_hash"].fillna("").astype(str).tolist())
development_domains.discard("")
development_templates.discard("")

h_domain = holdout_df["domain"].fillna("").astype(str)
h_template = holdout_df["template_hash"].fillna("").astype(str)
domain_seen = h_domain.isin(development_domains).to_numpy()
template_seen = h_template.isin(development_templates).to_numpy()
if "near_duplicate_to_development" not in holdout_df.columns:
    raise RuntimeError("near_duplicate_to_development fehlt.")
near_dup = holdout_df["near_duplicate_to_development"].fillna(False).astype(bool).to_numpy()

def balanced_available_indices(frame, mask, seed):
    sub = frame.loc[np.asarray(mask)].copy()
    n_each = min(int((sub.label == 0).sum()), int((sub.label == 1).sum()))
    p0 = sub[sub.label.eq(0)].sample(n=n_each, random_state=seed)
    p1 = sub[sub.label.eq(1)].sample(n=n_each, random_state=seed + 1)
    return np.sort(pd.concat([p0, p1]).index.to_numpy(dtype=np.int32))

domain_new_mask = (~domain_seen) & h_domain.ne("").to_numpy()
template_ood_mask = (~template_seen) & (~near_dup) & h_template.ne("").to_numpy()
domain_template_mask = domain_new_mask & template_ood_mask

stress_indices = {
    "TEMPORAL": np.arange(len(holdout_df), dtype=np.int32),
    "DOMAIN_OOD": balanced_available_indices(holdout_df, domain_new_mask, 108),
    "TEMPLATE_OOD": balanced_available_indices(holdout_df, template_ood_mask, 109),
    "DOMAIN_TEMPLATE_OOD": balanced_available_indices(holdout_df, domain_template_mask, 110),
}
for name, expected in EXPECTED_SCENARIO_ROWS.items():
    if len(stress_indices[name]) != expected:
        raise RuntimeError(f"{name}: {len(stress_indices[name])} statt {expected}")

iid_y = iid_df["label"].to_numpy(dtype=int)
iid_pos = np.flatnonzero(iid_y == 1)
iid_neg = np.flatnonzero(iid_y == 0)
n_each = min(len(iid_pos), len(iid_neg))
rng = np.random.default_rng(AP_BALANCE_SEED)
IID_BALANCED_IDX = np.sort(np.concatenate([
    rng.choice(iid_pos, n_each, replace=False),
    rng.choice(iid_neg, n_each, replace=False),
]).astype(np.int32))

SCENARIOS = {
    "IID_ORIGINAL": ("iid", np.arange(len(iid_df), dtype=np.int32)),
    "IID_BALANCED": ("iid", IID_BALANCED_IDX),
    "TEMPORAL": ("holdout", stress_indices["TEMPORAL"]),
    "DOMAIN_OOD": ("holdout", stress_indices["DOMAIN_OOD"]),
    "TEMPLATE_OOD": ("holdout", stress_indices["TEMPLATE_OOD"]),
    "DOMAIN_TEMPLATE_OOD": ("holdout", stress_indices["DOMAIN_TEMPLATE_OOD"]),
}
FINAL_STRESS = ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]

def load_token_split(folder):
    root = FREEZE_ROOT / "tokens" / folder
    return {
        "input_ids": np.load(root / "input_ids.npy", mmap_mode="r"),
        "attention_mask": np.load(root / "attention_mask.npy", mmap_mode="r"),
    }

TOKENS = {
    "train": load_token_split("train_4k"),
    "calibration": load_token_split("calibration"),
    "iid": load_token_split("iid_test"),
    "holdout": load_token_split("holdout_8k"),
}
expected_n = {
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid": len(iid_df),
    "holdout": len(holdout_df),
}
for split, arr in TOKENS.items():
    if arr["input_ids"].shape != (expected_n[split], MAX_LENGTH):
        raise RuntimeError(f"Token-Shape falsch: {split} {arr['input_ids'].shape}")

y_train = train_df["label"].to_numpy(dtype=int)
ycal = calibration_df["label"].to_numpy(dtype=int)

print({
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid": len(iid_df),
    "holdout": len(holdout_df),
    "stress": {k: len(v) for k, v in stress_indices.items()},
})


In [ ]:

# ============================================================
# 04 – Metriken, Scoring und E2E-Helfer
# ============================================================
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def threshold_for_target_fpr(y_true, score, target_fpr):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(score, dtype=float)
    neg = np.sort(s[y == 0])[::-1]
    allowed = int(math.floor(target_fpr * len(neg) + 1e-12))
    if allowed <= 0:
        return float(np.nextafter(neg[0], np.inf))
    if allowed >= len(neg):
        return float(-np.inf)
    return float(np.nextafter(neg[allowed], np.inf))

def metric_row(y, score, threshold):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(y, score)),
        "roc_auc": float(roc_auc_score(y, score)) if len(np.unique(y)) == 2 else np.nan,
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "empirical_fpr": float(fp / max(fp + tn, 1)),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }

def calibration_operating_points(ycal_, score):
    out = {}
    for target in TARGET_FPRS:
        thr = threshold_for_target_fpr(ycal_, score, target)
        m = metric_row(ycal_, score, thr)
        out[target] = {
            "threshold": thr,
            "calibration_empirical_fpr": m["empirical_fpr"],
            "calibration_recall": m["recall"],
        }
    return out

def scenario_score_view(iid_score, holdout_score, scenario):
    source, idx = SCENARIOS[scenario]
    if source == "iid":
        return (
            iid_df.iloc[idx]["label"].to_numpy(dtype=int),
            np.asarray(iid_score)[idx],
        )
    return (
        holdout_df.iloc[idx]["label"].to_numpy(dtype=int),
        np.asarray(holdout_score)[idx],
    )

class TokenSubsetDataset(Dataset):
    def __init__(self, arrays, labels, indices):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]
        self.labels = np.asarray(labels, dtype=np.int64)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        return (
            torch.tensor(self.ids[j], dtype=torch.long),
            torch.tensor(self.mask[j], dtype=torch.long),
            torch.tensor(self.labels[j], dtype=torch.long),
        )

class TokenAllDataset(Dataset):
    def __init__(self, arrays):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (
            torch.tensor(self.ids[i], dtype=torch.long),
            torch.tensor(self.mask[i], dtype=torch.long),
        )

@torch.no_grad()
def score_e2e_model(model, arrays, batch_size=E2E_SCORE_BATCH):
    ds = TokenAllDataset(arrays)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    model.eval()
    scores = []
    for ids, mask in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available(),
        ):
            logits = model(input_ids=ids, attention_mask=mask).logits
        scores.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(scores).astype(np.float32)

def train_refreshed_e2e(init_dir, seed, labeled_idx):
    last = None
    for bs in E2E_BATCH_CANDIDATES:
        try:
            set_all_seeds(seed + 99000)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = AutoModelForSequenceClassification.from_pretrained(
                str(init_dir),
                num_labels=2,
                local_files_only=True,
                ignore_mismatched_sizes=True,
            ).to(DEVICE)

            ds = TokenSubsetDataset(TOKENS["train"], y_train, labeled_idx)
            g = torch.Generator()
            g.manual_seed(seed + 99000)
            loader = DataLoader(
                ds,
                batch_size=bs,
                shuffle=True,
                generator=g,
                num_workers=2,
                pin_memory=torch.cuda.is_available(),
            )
            ga = max(1, 16 // bs)
            total_updates = math.ceil(len(loader) / ga) * E2E_EPOCHS
            opt = torch.optim.AdamW(
                model.parameters(),
                lr=E2E_LR,
                weight_decay=E2E_WEIGHT_DECAY,
            )
            sched = get_linear_schedule_with_warmup(
                opt,
                num_warmup_steps=int(round(total_updates * E2E_WARMUP_RATIO)),
                num_training_steps=total_updates,
            )
            scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
            started = time.perf_counter()

            for ep in range(E2E_EPOCHS):
                model.train()
                opt.zero_grad(set_to_none=True)
                losses = []
                for step, (ids, mask, labels) in enumerate(loader):
                    ids = ids.to(DEVICE, non_blocking=True)
                    mask = mask.to(DEVICE, non_blocking=True)
                    labels = labels.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast(
                        "cuda",
                        dtype=torch.float16,
                        enabled=torch.cuda.is_available(),
                    ):
                        loss = model(
                            input_ids=ids,
                            attention_mask=mask,
                            labels=labels,
                        ).loss / ga
                    scaler.scale(loss).backward()
                    losses.append(float(loss.detach().cpu()) * ga)
                    if (step + 1) % ga == 0 or step + 1 == len(loader):
                        scaler.unscale_(opt)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(opt)
                        scaler.update()
                        sched.step()
                        opt.zero_grad(set_to_none=True)
                print({
                    "refreshed_e2e_seed": seed,
                    "epoch": ep + 1,
                    "loss": float(np.mean(losses)),
                })

            return model, time.perf_counter() - started, bs, ga
        except RuntimeError as e:
            last = e
            if "out of memory" not in str(e).lower():
                raise
            try:
                del model
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError(f"Refreshed E2E failed seed={seed}") from last


In [ ]:

# ============================================================
# 05 – Korrigierter unlabeled-only SSL-Refresh
#      Pooler wird bewusst NICHT übertragen.
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    str(BASE_MODEL_DIR),
    local_files_only=True,
    use_fast=True,
)

class UnlabeledTokenSubsetDataset(Dataset):
    def __init__(self, arrays, indices):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        return {
            "input_ids": torch.tensor(self.ids[j], dtype=torch.long),
            "attention_mask": torch.tensor(self.mask[j], dtype=torch.long),
        }

ssl_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=SSL_REFRESH_MLM_PROB,
)

def has_hf_model(p):
    return (
        p.is_dir()
        and (p / "config.json").exists()
        and (
            (p / "model.safetensors").exists()
            or (p / "pytorch_model.bin").exists()
        )
    )

def ssl_refresh_encoder(seed, unlabeled_idx):
    out_dir = SSL_ROOT / f"seed_{seed}"
    cfg_path = out_dir / "ssl_refresh_config.json"
    if has_hf_model(out_dir) and cfg_path.exists():
        print({"ssl_refresh": seed, "status": "REUSE"})
        return out_dir

    set_all_seeds(seed + 88000)

    mlm = AutoModelForMaskedLM.from_pretrained(
        str(BASE_MODEL_DIR),
        local_files_only=True,
    ).to(DEVICE)

    # add_pooling_layer=False verhindert die zufällige Pooler-Erzeugung vollständig.
    dapt_encoder = AutoModel.from_pretrained(
        str(DAPT_DIRS[seed]),
        local_files_only=True,
        add_pooling_layer=False,
    )

    dapt_state = {
        k: v
        for k, v in dapt_encoder.state_dict().items()
        if not k.startswith("pooler.")
    }
    incompatible = mlm.roberta.load_state_dict(dapt_state, strict=False)
    bad_missing = [
        k for k in incompatible.missing_keys
        if not k.startswith("pooler.")
    ]
    bad_unexpected = [
        k for k in incompatible.unexpected_keys
        if not k.startswith("pooler.")
    ]
    if bad_missing or bad_unexpected:
        raise RuntimeError(
            f"DAPT->MLM mismatch seed={seed}: "
            f"missing={bad_missing}, unexpected={bad_unexpected}"
        )
    del dapt_encoder, dapt_state

    ds = UnlabeledTokenSubsetDataset(TOKENS["train"], unlabeled_idx)
    g = torch.Generator()
    g.manual_seed(seed + 88000)
    loader = DataLoader(
        ds,
        batch_size=SSL_REFRESH_BATCH,
        shuffle=True,
        generator=g,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        collate_fn=ssl_collator,
    )

    opt = torch.optim.AdamW(
        mlm.parameters(),
        lr=SSL_REFRESH_LR,
        weight_decay=SSL_REFRESH_WEIGHT_DECAY,
    )
    total_steps = len(loader) * SSL_REFRESH_EPOCHS
    sched = get_linear_schedule_with_warmup(
        opt,
        num_warmup_steps=int(round(0.10 * total_steps)),
        num_training_steps=total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    losses = []
    started = time.perf_counter()
    mlm.train()
    for ep in range(SSL_REFRESH_EPOCHS):
        for batch in loader:
            opt.zero_grad(set_to_none=True)
            ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16,
                enabled=torch.cuda.is_available(),
            ):
                loss = mlm(
                    input_ids=ids,
                    attention_mask=mask,
                    labels=labels,
                ).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mlm.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(float(loss.detach().cpu()))

    out_dir.mkdir(parents=True, exist_ok=True)
    mlm.roberta.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    cfg_path.write_text(json.dumps({
        "seed": seed,
        "objective": "TAPT-like continued MLM refresh",
        "source_encoder": str(DAPT_DIRS[seed]),
        "pool": "remaining unlabeled TRAIN rows only",
        "unlabeled_rows": int(len(unlabeled_idx)),
        "labels_used_for_ssl": False,
        "evaluation_rows_used": False,
        "epochs": SSL_REFRESH_EPOCHS,
        "batch_size": SSL_REFRESH_BATCH,
        "learning_rate": SSL_REFRESH_LR,
        "weight_decay": SSL_REFRESH_WEIGHT_DECAY,
        "mlm_probability": SSL_REFRESH_MLM_PROB,
        "mean_mlm_loss": float(np.mean(losses)),
        "seconds": float(time.perf_counter() - started),
        "pooler_transferred": False,
    }, indent=2), encoding="utf-8")

    del mlm, ds, loader, opt, sched, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print({
        "ssl_refresh": seed,
        "status": "COMPLETE",
        "unlabeled_rows": len(unlabeled_idx),
        "mean_loss": float(np.mean(losses)),
    })
    return out_dir


In [ ]:

# ============================================================
# 06 – NUR fehlenden SSL-Refresh + refreshed DAPT-E2E rechnen
# ============================================================
def refreshed_score_path(seed):
    return SCORE_ROOT / f"DAPT_SSL_REFRESH_seed{seed}_budget0p25.npz"

for seed in SEEDS:
    score_path = refreshed_score_path(seed)
    if score_path.exists():
        print({"seed": seed, "finish_only": "SCORES_REUSE"})
        continue

    key = f"{PRIMARY_STRATEGY}__seed{seed}"
    labeled = np.sort(np.asarray(selection_history[key]["0.25"], dtype=np.int32))
    unlabeled = np.setdiff1d(
        np.arange(len(y_train), dtype=np.int32),
        labeled,
    )

    # SSL sieht ausschließlich den ungelabelten Trainingsrest.
    refreshed_dir = ssl_refresh_encoder(seed, unlabeled)

    # Supervised Fine-Tuning sieht exakt dieselben AL-erworbenen 1000 Labels
    # wie DAPT_STATIC/T0 im bereits fertigen Loop.
    model, fit_seconds, bs, ga = train_refreshed_e2e(
        refreshed_dir,
        seed,
        labeled,
    )

    np.savez_compressed(
        score_path,
        cal=score_e2e_model(model, TOKENS["calibration"]),
        iid=score_e2e_model(model, TOKENS["iid"]),
        holdout=score_e2e_model(model, TOKENS["holdout"]),
        fit_seconds=np.asarray([fit_seconds], dtype=np.float64),
        batch_size=np.asarray([bs], dtype=np.int32),
        grad_accum=np.asarray([ga], dtype=np.int32),
        labeled_hash=np.asarray([
            hashlib.sha256(labeled.tobytes()).hexdigest()
        ]),
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    (AUDIT_ROOT / "FINISH_PROGRESS.json").write_text(
        json.dumps({
            "stage": "REFRESHED_SCORE_COMPLETE",
            "seed": seed,
            "utc": pd.Timestamp.utcnow().isoformat(),
        }, indent=2),
        encoding="utf-8",
    )

print({
    "refreshed_scores_complete": sum(refreshed_score_path(s).exists() for s in SEEDS),
    "expected": len(SEEDS),
})


In [ ]:

# ============================================================
# 07 – DAPT_STATIC vs DAPT_SSL_REFRESH:
#      direkter Detektor + identische B0-Kaskade
# ============================================================
rows = []

for seed in SEEDS:
    static = np.load(static_state_path(seed), allow_pickle=False)
    refreshed = np.load(refreshed_score_path(seed), allow_pickle=False)

    b0_ops = calibration_operating_points(ycal, static["b0_cal"])
    static_ops = calibration_operating_points(ycal, static["dapt_cal"])
    refresh_ops = calibration_operating_points(ycal, refreshed["cal"])

    models = {
        "DAPT_STATIC": {
            "cal": np.asarray(static["dapt_cal"]),
            "iid": np.asarray(static["dapt_iid"]),
            "holdout": np.asarray(static["dapt_holdout"]),
            "ops": static_ops,
        },
        "DAPT_SSL_REFRESH": {
            "cal": np.asarray(refreshed["cal"]),
            "iid": np.asarray(refreshed["iid"]),
            "holdout": np.asarray(refreshed["holdout"]),
            "ops": refresh_ops,
        },
    }

    for target in TARGET_FPRS:
        for scenario in SCENARIOS:
            y, bscore = scenario_score_view(
                static["b0_iid"],
                static["b0_holdout"],
                scenario,
            )
            bthr = b0_ops[target]["threshold"]
            bpred = bscore >= bthr
            neg = np.flatnonzero(~bpred)
            fnmask = (y == 1) & (~bpred)
            tp_stage1 = int(((y == 1) & bpred).sum())
            fn_stage1 = int(fnmask.sum())
            stage1_metrics = metric_row(y, bscore, bthr)

            for model_name, m in models.items():
                _, score = scenario_score_view(
                    m["iid"],
                    m["holdout"],
                    scenario,
                )
                direct = metric_row(
                    y,
                    score,
                    m["ops"][target]["threshold"],
                )

                for rf in REVIEW_FRACTIONS:
                    nr = max(1, int(math.ceil(rf * len(neg))))
                    review = neg[np.argsort(score[neg])[::-1]][:nr]
                    rescued = int(fnmask[review].sum())
                    assisted = (
                        (tp_stage1 + rescued)
                        / max(tp_stage1 + fn_stage1, 1)
                    )

                    rows.append({
                        "model": model_name,
                        "seed": seed,
                        "seed_group": "ORIGINAL5" if seed in ORIGINAL5 else "REPLICATION5",
                        "scenario": scenario,
                        "target_fpr": target,
                        "review_fraction": rf,
                        "stage1_recall": stage1_metrics["recall"],
                        "stage1_fpr": stage1_metrics["empirical_fpr"],
                        "direct_recall": direct["recall"],
                        "direct_fpr": direct["empirical_fpr"],
                        "direct_precision": direct["precision"],
                        "direct_f1": direct["f1"],
                        "direct_ap": direct["average_precision"],
                        "direct_auc": direct["roc_auc"],
                        "review_n": nr,
                        "review_fraction_all": nr / len(y),
                        "review_precision": float((y[review] == 1).mean()),
                        "rescued_stage1_fn": rescued,
                        "capture_rate": rescued / max(fn_stage1, 1),
                        "assisted_recall": assisted,
                    })

RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(
    OUT_ROOT / "LOOP_SSL_refresh_ablation.csv",
    index=False,
)

expected = 2 * len(SEEDS) * len(SCENARIOS) * len(TARGET_FPRS) * len(REVIEW_FRACTIONS)
if len(RESULTS) != expected:
    raise RuntimeError(f"Ergebniscoverage {len(RESULTS)} != {expected}")

print({
    "ssl_refresh_ablation_rows": len(RESULTS),
    "expected": expected,
})


In [ ]:

# ============================================================
# 08 – Gepaarte N10-Statistik
# ============================================================
def exact_signflip(diff):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    obs = abs(diff.mean())
    vals = []
    for signs in product([-1.0, 1.0], repeat=len(diff)):
        vals.append(abs(np.mean(diff * np.asarray(signs))))
    return float(np.mean(np.asarray(vals) >= obs - 1e-15))

def paired_stats(diff):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    n = len(diff)
    mean = float(diff.mean())
    sd = float(diff.std(ddof=1)) if n > 1 else np.nan
    sem = sd / math.sqrt(n) if n > 1 else np.nan
    crit = float(student_t.ppf(0.975, n - 1)) if n > 1 else np.nan
    return {
        "n_seeds": n,
        "mean_difference": mean,
        "sd_difference": sd,
        "ci95_low": mean - crit * sem if n > 1 else np.nan,
        "ci95_high": mean + crit * sem if n > 1 else np.nan,
        "exact_signflip_p_two_sided": exact_signflip(diff),
        "cohen_dz": mean / sd if n > 1 and sd > 0 else np.nan,
        "wins": int((diff > 1e-12).sum()),
        "ties": int((np.abs(diff) <= 1e-12).sum()),
        "losses": int((diff < -1e-12).sum()),
    }

stat_rows = []
for group_name, group_seeds in {
    "ORIGINAL5": ORIGINAL5,
    "REPLICATION5": REPLICATION5,
    "POOLED10": SEEDS,
}.items():
    for scenario in FINAL_STRESS:
        for metric in [
            "direct_recall",
            "direct_fpr",
            "direct_ap",
            "assisted_recall",
            "review_precision",
            "capture_rate",
        ]:
            a = RESULTS[
                (RESULTS.model == "DAPT_SSL_REFRESH")
                & (RESULTS.seed.isin(group_seeds))
                & (RESULTS.scenario == scenario)
                & (RESULTS.target_fpr == PRIMARY_FPR)
                & (RESULTS.review_fraction == PRIMARY_REVIEW)
            ].set_index("seed")
            b = RESULTS[
                (RESULTS.model == "DAPT_STATIC")
                & (RESULTS.seed.isin(group_seeds))
                & (RESULTS.scenario == scenario)
                & (RESULTS.target_fpr == PRIMARY_FPR)
                & (RESULTS.review_fraction == PRIMARY_REVIEW)
            ].set_index("seed")
            diff = (
                a.loc[group_seeds, metric].to_numpy()
                - b.loc[group_seeds, metric].to_numpy()
            )
            stat_rows.append({
                "seed_group": group_name,
                "contrast": "DAPT_SSL_REFRESH_minus_DAPT_STATIC",
                "scenario": scenario,
                "metric": metric,
                "target_fpr": PRIMARY_FPR,
                "review_fraction": PRIMARY_REVIEW,
                **paired_stats(diff),
            })

STATS = pd.DataFrame(stat_rows)
STATS.to_csv(
    OUT_ROOT / "LOOP_SSL_refresh_statistics.csv",
    index=False,
)

print(
    STATS[
        (STATS.seed_group == "POOLED10")
        & STATS.metric.isin(["direct_recall", "assisted_recall"])
    ][[
        "scenario",
        "metric",
        "mean_difference",
        "ci95_low",
        "ci95_high",
        "exact_signflip_p_two_sided",
        "wins",
        "ties",
        "losses",
    ]].to_string(index=False)
)


In [ ]:

# ============================================================
# 09 – Alte Loop-Ergebnisse übernehmen + Thesis-Tabellen/Figuren
# ============================================================
# Kleine bereits fertige Dateien werden 1:1 übernommen.
for name in [
    "LOOP_results_long.csv",
    "LOOP_summary.csv",
    "LOOP_statistics_n10.csv",
    "LOOP_acquisition_audit.csv",
    "LOOP_annotation_efficiency.csv",
    "LOOP_T0_DAPT_same_AL_labels.csv",
    "LOOP_T0_DAPT_same_AL_labels_statistics.csv",
]:
    shutil.copy2(LOOP_SOURCE / name, OUT_ROOT / name)

LOOP_RESULTS = pd.read_csv(OUT_ROOT / "LOOP_results_long.csv")
LOOP_SUMMARY = pd.read_csv(OUT_ROOT / "LOOP_summary.csv")
T0_DAPT = pd.read_csv(OUT_ROOT / "LOOP_T0_DAPT_same_AL_labels.csv")
T0_DAPT_STATS = pd.read_csv(OUT_ROOT / "LOOP_T0_DAPT_same_AL_labels_statistics.csv")

# Tabellen
primary_refresh = RESULTS[
    RESULTS.scenario.isin(FINAL_STRESS)
    & RESULTS.target_fpr.eq(PRIMARY_FPR)
    & RESULTS.review_fraction.eq(PRIMARY_REVIEW)
]
primary_refresh.to_csv(TABLE_ROOT / "TABLE01_SSL_refresh_primary.csv", index=False)

STATS[
    STATS.scenario.isin(FINAL_STRESS)
].to_csv(TABLE_ROOT / "TABLE02_SSL_refresh_statistics.csv", index=False)

T0_DAPT.to_csv(TABLE_ROOT / "TABLE03_T0_DAPT_same_AL_labels.csv", index=False)
T0_DAPT_STATS.to_csv(TABLE_ROOT / "TABLE04_T0_DAPT_same_AL_labels_statistics.csv", index=False)

LOOP_SUMMARY.to_csv(TABLE_ROOT / "TABLE05_productive_loop_summary.csv", index=False)

# Kompakte Systemübersicht über den produktiven Hauptpfad.
summary_refresh = (
    primary_refresh
    .groupby(["model", "scenario"], as_index=False)
    .agg(
        direct_recall_mean=("direct_recall", "mean"),
        direct_fpr_mean=("direct_fpr", "mean"),
        direct_ap_mean=("direct_ap", "mean"),
        review_precision_mean=("review_precision", "mean"),
        assisted_recall_mean=("assisted_recall", "mean"),
        capture_rate_mean=("capture_rate", "mean"),
    )
)
summary_refresh.to_csv(TABLE_ROOT / "TABLE06_SSL_refresh_summary.csv", index=False)

manifest = []

def savefig(fig, name, purpose):
    png = FIG_ROOT / f"{name}.png"
    pdf = FIG_ROOT / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(png, dpi=220, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    manifest.append({
        "figure": name,
        "purpose": purpose,
        "png": png.name,
        "pdf": pdf.name,
    })

# FIG1: direkter Recall
fig, ax = plt.subplots(figsize=(8.8, 5.2))
x = np.arange(len(FINAL_STRESS))
for model in ["DAPT_STATIC", "DAPT_SSL_REFRESH"]:
    d = (
        primary_refresh[primary_refresh.model == model]
        .groupby("scenario", as_index=False)
        .agg(recall=("direct_recall", "mean"))
        .set_index("scenario")
        .loc[FINAL_STRESS]
    )
    ax.plot(x, d.recall, marker="o", label=model)
ax.set_xticks(x)
ax.set_xticklabels(FINAL_STRESS, rotation=20, ha="right")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.set_title("Zusätzlicher unlabeled SSL-Refresh – direkter Detektor")
ax.legend()
ax.grid(True, alpha=0.25)
savefig(fig, "FIG01_SSL_refresh_direct_recall", "DAPT static vs zusätzlicher MLM-Refresh")

# FIG2: Kaskade
fig, ax = plt.subplots(figsize=(8.8, 5.2))
for model in ["DAPT_STATIC", "DAPT_SSL_REFRESH"]:
    d = (
        primary_refresh[primary_refresh.model == model]
        .groupby("scenario", as_index=False)
        .agg(recall=("assisted_recall", "mean"))
        .set_index("scenario")
        .loc[FINAL_STRESS]
    )
    ax.plot(x, d.recall, marker="o", label=model)
ax.set_xticks(x)
ax.set_xticklabels(FINAL_STRESS, rotation=20, ha="right")
ax.set_ylabel("Review-gestützter Recall")
ax.set_ylim(0, 1)
ax.set_title("Zusätzlicher unlabeled SSL-Refresh – produktive Kaskade")
ax.legend()
ax.grid(True, alpha=0.25)
savefig(fig, "FIG02_SSL_refresh_assisted_recall", "Einfluss des SSL-Refresh auf die Kaskade")

# FIG3: bisheriger Active-Learning-Loop
primary_loop = LOOP_SUMMARY[
    LOOP_SUMMARY.scenario.isin(FINAL_STRESS)
    & LOOP_SUMMARY.target_fpr.eq(PRIMARY_FPR)
    & LOOP_SUMMARY.review_fraction_of_stage1_negatives.eq(PRIMARY_REVIEW)
]
fig, ax = plt.subplots(figsize=(8.8, 5.2))
for strategy in [
    "RANDOM_GLOBAL",
    "UNCERTAINTY_GLOBAL",
    "UNCERTAINTY_POST_CASCADE",
]:
    d = (
        primary_loop[primary_loop.strategy == strategy]
        .groupby("label_budget", as_index=False)
        .agg(recall=("assisted_recall_mean", "mean"))
        .sort_values("label_budget")
    )
    ax.plot(100 * d.label_budget, d.recall, marker="o", label=strategy)
ax.set_xlabel("Gelabelte Trainingsdaten [%]")
ax.set_ylabel("Mittlerer review-gestützter Recall")
ax.set_xticks([10, 15, 20, 25])
ax.set_ylim(0, 1)
ax.set_title("Produktiver Feedback-Loop")
ax.legend()
ax.grid(True, alpha=0.25)
savefig(fig, "FIG03_productive_feedback_loop", "Active-Learning-Feedback im Gesamtsystem")

pd.DataFrame(manifest).to_csv(FIG_ROOT / "figure_manifest.csv", index=False)

print({
    "tables": len(list(TABLE_ROOT.glob("*.csv"))),
    "figures_png": len(list(FIG_ROOT.glob("*.png"))),
})


In [ ]:

# ============================================================
# 10 – Completion Audit + kleines Analyse-ZIP
# ============================================================
checks = {
    "old_loop_retrained": False,
    "n10_retrained": False,
    "old_static_states_validated": len(SEEDS),
    "refreshed_scores": sum(refreshed_score_path(s).exists() for s in SEEDS),
    "ssl_refresh_ablation_rows": len(RESULTS),
    "statistics_rows": len(STATS),
    "performance_based_seed_exclusions": [],
    "ssl_refresh_uses_labels": False,
    "ssl_refresh_uses_evaluation_data": False,
    "e2e_uses_same_25pct_AL_labels_as_static_loop": True,
}

if checks["refreshed_scores"] != 10:
    raise RuntimeError("Nicht alle 10 refreshed Scores vorhanden.")

completion = {
    "status": "COMPLETE",
    "run": "PRODUCTIVE_LOOP_FINISH_ONLY_SSL_REFRESH",
    "source_loop": str(LOOP_SOURCE),
    "source_n10": str(N10_SOURCE),
    "seeds": SEEDS,
    "primary_target_fpr": PRIMARY_FPR,
    "primary_review_fraction": PRIMARY_REVIEW,
    "scientific_role": (
        "explorative lifecycle extension; primary SSL generalization "
        "evidence remains the frozen N10 T0-vs-DAPT study"
    ),
    "ssl_refresh": {
        "objective": "TAPT-like continued MLM",
        "pool": "remaining unlabeled original TRAIN rows",
        "labels_used": False,
        "evaluation_data_used": False,
    },
    "checks": checks,
}
(OUT_ROOT / "PRODUCTIVE_FINISH_ONLY_COMPLETE.json").write_text(
    json.dumps(completion, indent=2),
    encoding="utf-8",
)

analysis = OUT_ROOT / "ANALYSIS_PACKAGE"
if analysis.exists():
    shutil.rmtree(analysis)
analysis.mkdir(parents=True)

for p in OUT_ROOT.glob("*.csv"):
    shutil.copy2(p, analysis / p.name)
for p in OUT_ROOT.glob("*.json"):
    shutil.copy2(p, analysis / p.name)
for folder in ["tables", "figures", "audit"]:
    src = OUT_ROOT / folder
    if src.exists():
        shutil.copytree(src, analysis / folder, dirs_exist_ok=True)

archive = shutil.make_archive(
    "/kaggle/working/phreshphish_PRODUCTIVE_LOOP_FINISH_ONLY_ANALYSIS",
    "zip",
    root_dir=analysis,
)

print(json.dumps(completion, indent=2))
print({"ANALYSIS_ZIP": archive})


## Ergebnisinterpretation

Für die Bachelorarbeit bleibt die Hierarchie unverändert:

1. **Primärer SSL-Befund:** N10-Vergleich `T0_E2E` vs. `DAPT_E2E` unter Distribution Shift
   und unterschiedlichen Labelbudgets.
2. **Produktive Erweiterung:** Kaskade + Active Learning als Lifecycle des Artefakts.
3. **Zusatzablation dieses Finish-Laufs:** Prüft, ob ein weiterer unlabeled-only MLM-Refresh
   nach der aktiven Labelakquisition über das bereits vorhandene DAPT hinaus zusätzlichen
   Nutzen erzeugt.

Ein positiver SSL-Refresh-Effekt ist ein Zusatzbefund. Ein Null- oder Negativergebnis
widerlegt den ursprünglichen DAPT-Befund nicht, da hier eine andere, nachgelagerte
Adaptionsstufe und ein anderer unlabeled Pool untersucht werden.
